# 🎓 Üniversite Ders Asistanı — RAG Sistemi
**OpenAI GPT-4 + text-embedding-3-small + FAISS**

Bu notebook, üniversite ders PDF'lerinden anlam tabanlı soru-cevap sistemi kurar.

---
### Mimari
```
PDF Dosyaları → PyMuPDF Parsing → Chunking (300 token / 50 overlap)
     → spaCy NER → OpenAI Embedding → FAISS Index
     → Soru → Embedding → Similarity Search → GPT-4 Yanıt
```

## 📦 1. Kurulum ve Bağımlılıklar

In [ ]:
# Gerekli paketleri kur
!pip install openai pymupdf tiktoken faiss-cpu spacy python-dotenv numpy pandas
!python -m spacy download tr_core_news_sm  # Türkçe NER modeli
# İngilizce dersler için: !python -m spacy download en_core_web_sm

In [ ]:
import os
import json
import re
import numpy as np
import pandas as pd
import fitz  # PyMuPDF
import tiktoken
import faiss
import spacy
from openai import OpenAI
from pathlib import Path
from typing import List, Dict, Tuple
from dataclasses import dataclass, field
from IPython.display import display, Markdown

print("✅ Tüm kütüphaneler yüklendi.")

## 🔑 2. Yapılandırma

In [ ]:
# ─────────────────────────────────────────────
# API Anahtarı — .env dosyasından veya direkt girin
# ─────────────────────────────────────────────
# Yöntem 1: Doğrudan girin (test için)
OPENAI_API_KEY = "sk-..."  # <-- kendi anahtarınızı girin

# Yöntem 2: .env dosyasından oku
# from dotenv import load_dotenv
# load_dotenv()
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)

# ─────────────────────────────────────────────
# Sistem Ayarları
# ─────────────────────────────────────────────
CONFIG = {
    "embedding_model": "text-embedding-3-small",   # Ucuz & güçlü
    "llm_model":       "gpt-4o",                    # GPT-4o (daha hızlı)
    "chunk_size":      300,                          # token
    "chunk_overlap":   50,                           # token
    "top_k":           5,                            # kaç chunk getirilsin
    "pdf_dir":         "./pdfs",                     # PDF klasörü
    "index_path":      "./faiss_index.bin",
    "metadata_path":   "./chunks_metadata.json",
    "spacy_model":     "tr_core_news_sm",            # Türkçe; İngilizce için: en_core_web_sm
}

# PDF klasörünü oluştur (yoksa)
Path(CONFIG["pdf_dir"]).mkdir(exist_ok=True)
print("✅ Yapılandırma tamamlandı.")
print(f"   Embedding : {CONFIG['embedding_model']}")
print(f"   LLM       : {CONFIG['llm_model']}")
print(f"   Chunk     : {CONFIG['chunk_size']} token / {CONFIG['chunk_overlap']} overlap")

## 📄 3. PDF Parsing — PyMuPDF

In [ ]:
@dataclass
class ParsedPage:
    """Tek bir PDF sayfasından çıkarılan veri."""
    source_file: str
    page_number: int
    heading: str
    text: str
    entities: List[Dict] = field(default_factory=list)


def clean_text(text: str) -> str:
    """Ham PDF metnini temizle."""
    text = re.sub(r'\s+', ' ', text)          # Fazla boşlukları sil
    text = re.sub(r'([a-z])-\n([a-z])', r'\1\2', text)  # Tire satır kırma
    text = re.sub(r'\n+', ' ', text)           # Satır sonlarını boşluğa çevir
    text = text.strip()
    return text


def extract_heading(page: fitz.Page) -> str:
    """Sayfadaki en büyük font boyutlu metni başlık olarak al."""
    blocks = page.get_text("dict")["blocks"]
    candidates = []
    for block in blocks:
        if block["type"] == 0:  # text block
            for line in block["lines"]:
                for span in line["spans"]:
                    if span["size"] >= 14 and len(span["text"].strip()) > 3:
                        candidates.append((span["size"], span["text"].strip()))
    if candidates:
        candidates.sort(reverse=True)
        return candidates[0][1][:100]  # max 100 karakter
    return "Başlık Yok"


def parse_pdf(pdf_path: str) -> List[ParsedPage]:
    """Tek bir PDF'i sayfa sayfa parse et."""
    pages = []
    doc = fitz.open(pdf_path)
    file_name = Path(pdf_path).name
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        raw_text = page.get_text("text")
        cleaned  = clean_text(raw_text)
        heading  = extract_heading(page)
        
        if len(cleaned) > 50:  # Boş / çok kısa sayfaları atla
            pages.append(ParsedPage(
                source_file=file_name,
                page_number=page_num + 1,
                heading=heading,
                text=cleaned
            ))
    doc.close()
    return pages


def parse_all_pdfs(pdf_dir: str) -> List[ParsedPage]:
    """Klasördeki tüm PDF'leri parse et."""
    all_pages = []
    pdf_files = list(Path(pdf_dir).glob("*.pdf"))
    
    if not pdf_files:
        print(f"⚠️  '{pdf_dir}' klasöründe PDF bulunamadı!")
        print("   Lütfen PDF dosyalarınızı bu klasöre kopyalayın.")
        return []
    
    print(f"📂 {len(pdf_files)} PDF dosyası bulundu.")
    for pdf_path in pdf_files:
        try:
            pages = parse_pdf(str(pdf_path))
            all_pages.extend(pages)
            print(f"   ✅ {pdf_path.name} → {len(pages)} sayfa")
        except Exception as e:
            print(f"   ❌ {pdf_path.name} HATA: {e}")
    
    print(f"\n📊 Toplam: {len(all_pages)} sayfa parse edildi.")
    return all_pages


# ── ÇALIŞTIR ──────────────────────────────────
parsed_pages = parse_all_pdfs(CONFIG["pdf_dir"])

# Örnek çıktı
if parsed_pages:
    sample = parsed_pages[0]
    print(f"\n🔍 Örnek Sayfa:")
    print(f"   Dosya   : {sample.source_file}")
    print(f"   Sayfa   : {sample.page_number}")
    print(f"   Başlık  : {sample.heading}")
    print(f"   Metin   : {sample.text[:200]}...")

## ✂️ 4. Chunking — 300 Token / 50 Overlap

In [ ]:
@dataclass
class Chunk:
    """Tek bir metin chunk'ı ve metadata."""
    chunk_id:    int
    source_file: str
    page_number: int
    heading:     str
    text:        str
    token_count: int
    entities:    List[Dict] = field(default_factory=list)
    embedding:   List[float] = field(default_factory=list)


def tokenize(text: str, encoding_name: str = "cl100k_base") -> List[int]:
    enc = tiktoken.get_encoding(encoding_name)
    return enc.encode(text)


def detokenize(tokens: List[int], encoding_name: str = "cl100k_base") -> str:
    enc = tiktoken.get_encoding(encoding_name)
    return enc.decode(tokens)


def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> List[str]:
    """Token bazlı sliding-window chunking."""
    tokens = tokenize(text)
    chunks = []
    start  = 0
    
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunk_tokens = tokens[start:end]
        chunks.append(detokenize(chunk_tokens))
        if end == len(tokens):
            break
        start += chunk_size - overlap
    
    return chunks


def create_chunks(pages: List[ParsedPage],
                  chunk_size: int = 300,
                  overlap: int    = 50) -> List[Chunk]:
    """Tüm sayfaları chunk'lara böl."""
    all_chunks = []
    chunk_id   = 0
    
    for page in pages:
        # Başlık-aware: başlığı her chunk'ın önüne ekle
        enriched_text = f"[{page.heading}] {page.text}"
        text_chunks   = chunk_text(enriched_text, chunk_size, overlap)
        
        for chunk_text_part in text_chunks:
            token_count = len(tokenize(chunk_text_part))
            all_chunks.append(Chunk(
                chunk_id=chunk_id,
                source_file=page.source_file,
                page_number=page.page_number,
                heading=page.heading,
                text=chunk_text_part,
                token_count=token_count
            ))
            chunk_id += 1
    
    return all_chunks


# ── ÇALIŞTIR ──────────────────────────────────
chunks = create_chunks(parsed_pages, CONFIG["chunk_size"], CONFIG["chunk_overlap"])
print(f"✅ Toplam chunk sayısı: {len(chunks)}")

# Chunk istatistikleri
if chunks:
    token_counts = [c.token_count for c in chunks]
    print(f"   Ort. token/chunk : {np.mean(token_counts):.0f}")
    print(f"   Min token/chunk  : {min(token_counts)}")
    print(f"   Max token/chunk  : {max(token_counts)}")
    print(f"\n🔍 Örnek Chunk #0:")
    print(chunks[0].text[:300])

## 🏷️ 5. NER — spaCy ile Varlık Tanıma

In [ ]:
# spaCy modelini yükle
try:
    nlp = spacy.load(CONFIG["spacy_model"])
    print(f"✅ spaCy modeli yüklendi: {CONFIG['spacy_model']}")
except OSError:
    print(f"⚠️  Model bulunamadı. Yükleniyor...")
    os.system(f"python -m spacy download {CONFIG['spacy_model']}")
    nlp = spacy.load(CONFIG["spacy_model"])


# Özel kural tabanlı etiketleyici ekle (akademik terimler için)
from spacy.pipeline import EntityRuler

ruler = nlp.add_pipe("entity_ruler", before="ner")
patterns = [
    # Algoritma / Konu etiketleri
    {"label": "TOPIC", "pattern": "binary search"},
    {"label": "TOPIC", "pattern": "sorting"},
    {"label": "TOPIC", "pattern": "recursion"},
    {"label": "TOPIC", "pattern": "dynamic programming"},
    {"label": "TOPIC", "pattern": "makine öğrenmesi"},
    {"label": "TOPIC", "pattern": "yapay zeka"},
    # Sınav tarihleri
    {"label": "EXAM",  "pattern": [{"LOWER": {"IN": ["midterm", "vize", "final", "quiz"]}}]},
]
ruler.add_patterns(patterns)


def extract_entities(text: str) -> List[Dict]:
    """Metinden NER varlıklarını çıkar."""
    doc = nlp(text[:1000])  # spaCy için ilk 1000 karakter yeterli
    entities = []
    seen = set()
    for ent in doc.ents:
        key = (ent.text.lower(), ent.label_)
        if key not in seen:
            entities.append({"text": ent.text, "label": ent.label_})
            seen.add(key)
    return entities


def apply_ner_to_chunks(chunks: List[Chunk]) -> List[Chunk]:
    """Her chunk'a NER uygula."""
    print("🏷️  NER uygulanıyor...")
    for i, chunk in enumerate(chunks):
        chunk.entities = extract_entities(chunk.text)
        if (i + 1) % 100 == 0:
            print(f"   {i+1}/{len(chunks)} chunk işlendi")
    print(f"✅ NER tamamlandı. {len(chunks)} chunk etiketlendi.")
    return chunks


# ── ÇALIŞTIR ──────────────────────────────────
chunks = apply_ner_to_chunks(chunks)

# NER istatistikleri
all_labels = {}
for c in chunks:
    for ent in c.entities:
        all_labels[ent["label"]] = all_labels.get(ent["label"], 0) + 1

print("\n📊 Bulunan Varlık Türleri:")
for label, count in sorted(all_labels.items(), key=lambda x: -x[1]):
    print(f"   {label:12s}: {count}")

## 🔢 6. Embedding — OpenAI text-embedding-3-small

In [ ]:
import time

def get_embeddings_batch(texts: List[str],
                          model: str = "text-embedding-3-small",
                          batch_size: int = 100) -> List[List[float]]:
    """OpenAI API ile batch embedding al."""
    all_embeddings = []
    total_batches  = (len(texts) + batch_size - 1) // batch_size
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        batch_num = i // batch_size + 1
        
        try:
            response = client.embeddings.create(
                input=batch,
                model=model
            )
            batch_embeddings = [item.embedding for item in response.data]
            all_embeddings.extend(batch_embeddings)
            print(f"   Batch {batch_num}/{total_batches} tamamlandı ({len(batch)} metin)")
            time.sleep(0.5)  # Rate limit koruması
        except Exception as e:
            print(f"   ❌ Batch {batch_num} HATA: {e}")
            # Hata durumunda sıfır vektör ekle
            all_embeddings.extend([[0.0] * 1536] * len(batch))
    
    return all_embeddings


def embed_chunks(chunks: List[Chunk]) -> List[Chunk]:
    """Tüm chunk'ları embed et."""
    print(f"🔢 {len(chunks)} chunk embed ediliyor...")
    texts = [c.text for c in chunks]
    embeddings = get_embeddings_batch(texts, model=CONFIG["embedding_model"])
    
    for chunk, embedding in zip(chunks, embeddings):
        chunk.embedding = embedding
    
    print(f"✅ Embedding tamamlandı. Boyut: {len(embeddings[0])}D")
    return chunks


# ── ÇALIŞTIR ──────────────────────────────────
# ⚠️ API maliyeti: ~10.000 chunk × 300 token ≈ 3M token ≈ $0.06 (text-embedding-3-small)
chunks = embed_chunks(chunks)
print(f"\n💡 Örnek embedding boyutu: {len(chunks[0].embedding)} boyut")

## 🗄️ 7. FAISS Vector Store — İndeks Oluşturma

In [ ]:
def build_faiss_index(chunks: List[Chunk]) -> faiss.Index:
    """FAISS L2 index oluştur."""
    embeddings = np.array([c.embedding for c in chunks], dtype=np.float32)
    
    # Normalize (cosine similarity için)
    faiss.normalize_L2(embeddings)
    
    dim   = embeddings.shape[1]  # 1536
    index = faiss.IndexFlatIP(dim)  # Inner Product = cosine similarity (normalize sonrası)
    index.add(embeddings)
    
    print(f"✅ FAISS index oluşturuldu.")
    print(f"   Boyut : {dim}D")
    print(f"   Vektör: {index.ntotal} adet")
    return index


def save_index(index: faiss.Index, chunks: List[Chunk],
               index_path: str, metadata_path: str):
    """İndeks ve metadata'yı diske kaydet."""
    faiss.write_index(index, index_path)
    
    # Metadata (embedding hariç — çok büyük)
    metadata = [{
        "chunk_id":    c.chunk_id,
        "source_file": c.source_file,
        "page_number": c.page_number,
        "heading":     c.heading,
        "text":        c.text,
        "token_count": c.token_count,
        "entities":    c.entities
    } for c in chunks]
    
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    
    print(f"✅ İndeks kaydedildi → {index_path}")
    print(f"✅ Metadata kaydedildi → {metadata_path}")


def load_index(index_path: str, metadata_path: str) -> Tuple[faiss.Index, List[Dict]]:
    """Kaydedilmiş indeks ve metadata'yı yükle."""
    index    = faiss.read_index(index_path)
    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    print(f"✅ İndeks yüklendi: {index.ntotal} vektör")
    return index, metadata


# ── ÇALIŞTIR ──────────────────────────────────
index = build_faiss_index(chunks)
save_index(index, chunks, CONFIG["index_path"], CONFIG["metadata_path"])

## 🔍 8. Retrieval — Benzer Chunk Bulma

In [ ]:
def get_query_embedding(query: str, model: str = "text-embedding-3-small") -> np.ndarray:
    """Sorgu metnini embed et."""
    response = client.embeddings.create(input=[query], model=model)
    vec = np.array([response.data[0].embedding], dtype=np.float32)
    faiss.normalize_L2(vec)
    return vec


def retrieve(query: str,
             index: faiss.Index,
             metadata: List[Dict],
             top_k: int = 5) -> List[Dict]:
    """En alakalı top_k chunk'ı getir."""
    query_vec = get_query_embedding(query, CONFIG["embedding_model"])
    scores, indices = index.search(query_vec, top_k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < len(metadata):
            result = metadata[idx].copy()
            result["similarity_score"] = float(score)
            results.append(result)
    
    return results


# ── TEST ──────────────────────────────────────
test_query = "Binary search nasıl çalışır?"
# index ve metadata yukarıda oluşturulmuş olmalı
results = retrieve(test_query, index, 
                   [vars(c) for c in chunks],  # Chunk'ları dict'e çevir
                   top_k=CONFIG["top_k"])

print(f"🔍 Sorgu: '{test_query}'")
print(f"   {len(results)} chunk getirildi.\n")
for i, r in enumerate(results):
    print(f"   [{i+1}] Skor={r['similarity_score']:.3f} | {r['source_file']} s.{r['page_number']}")
    print(f"        Başlık: {r['heading']}")
    print(f"        Metin : {r['text'][:150]}...\n")

## 🤖 9. Generation — GPT-4 ile Cevap Üretme

In [ ]:
SYSTEM_PROMPT = """Sen bir üniversite ders asistanısın. 
Sana verilen ders notları ve slaytlardan elde edilen bağlam bilgisine dayanarak 
öğrencilerin sorularını Türkçe olarak doğru ve anlaşılır biçimde yanıtla.

Kurallar:
1. Sadece verilen bağlamda bulunan bilgileri kullan.
2. Bağlamda yoksa 'Bu bilgi ders materyallerinde bulunamadı.' de.
3. Kaynağı (dosya adı ve sayfa numarası) belirt.
4. Kısa, net ve öğrenci dostu bir dil kullan."""


def format_context(retrieved_chunks: List[Dict]) -> str:
    """Getirilen chunk'ları GPT için bağlam metnine dönüştür."""
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        context_parts.append(
            f"[Kaynak {i}: {chunk['source_file']}, Sayfa {chunk['page_number']}]\n"
            f"{chunk['text']}"
        )
    return "\n\n".join(context_parts)


def generate_answer(query: str,
                    retrieved_chunks: List[Dict],
                    model: str = "gpt-4o") -> str:
    """GPT-4 ile son cevabı üret."""
    context = format_context(retrieved_chunks)
    
    user_message = f"""Bağlam:
{context}

---
Soru: {query}"""
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system",  "content": SYSTEM_PROMPT},
            {"role": "user",    "content": user_message}
        ],
        temperature=0.2,
        max_tokens=800
    )
    
    return response.choices[0].message.content


print("✅ Generation modülü hazır.")

## 🔗 10. Tam RAG Pipeline

In [ ]:
# Metadata'yı yeniden yükle (daha önce kaydedilmişse)
# index, metadata = load_index(CONFIG["index_path"], CONFIG["metadata_path"])

def rag_query(question: str,
              index: faiss.Index,
              metadata: List[Dict],
              top_k: int = 5,
              verbose: bool = True) -> str:
    """Tam RAG pipeline: sorgu → retrieval → generation."""
    if verbose:
        print(f"📥 Sorgu: {question}")
        print("   🔍 Benzer chunk'lar aranıyor...")
    
    # 1. Retrieve
    retrieved = retrieve(question, index, metadata, top_k=top_k)
    
    if verbose:
        print(f"   📄 {len(retrieved)} kaynak bulundu:")
        for r in retrieved:
            print(f"      → {r['source_file']} s.{r['page_number']} "
                  f"(benzerlik: {r['similarity_score']:.3f})")
        print("   🤖 GPT-4 cevap üretiyor...")
    
    # 2. Generate
    answer = generate_answer(question, retrieved, model=CONFIG["llm_model"])
    
    return answer


# ─────────────────────────────────────────────
# Chunks metadata'sını hazırla
chunks_metadata = [{
    "chunk_id":         c.chunk_id,
    "source_file":      c.source_file,
    "page_number":      c.page_number,
    "heading":          c.heading,
    "text":             c.text,
    "token_count":      c.token_count,
    "entities":         c.entities,
    "similarity_score": 0.0
} for c in chunks]

print("✅ RAG pipeline hazır!")

## 🧪 11. Test Soruları

In [ ]:
# Test 1: Kavramsal soru
q1 = "Binary search nasıl çalışır?"
answer1 = rag_query(q1, index, chunks_metadata)
display(Markdown(f"### ❓ {q1}\n\n**Cevap:**\n\n{answer1}"))

In [ ]:
# Test 2: Sınav odaklı soru
q2 = "Midtermde hangi konular çıktı?"
answer2 = rag_query(q2, index, chunks_metadata)
display(Markdown(f"### ❓ {q2}\n\n**Cevap:**\n\n{answer2}"))

In [ ]:
# Test 3: Karmaşıklık analizi sorusu
q3 = "Quick sort'un zaman karmaşıklığı nedir?"
answer3 = rag_query(q3, index, chunks_metadata)
display(Markdown(f"### ❓ {q3}\n\n**Cevap:**\n\n{answer3}"))

## 💬 12. İnteraktif Soru-Cevap Arayüzü

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

# Basit chat arayüzü
chat_history = []

question_input = widgets.Textarea(
    placeholder='Sorunuzu buraya yazın...',
    layout=widgets.Layout(width='100%', height='80px')
)
ask_button = widgets.Button(
    description='Sor 🔍',
    button_style='primary',
    layout=widgets.Layout(width='120px')
)
output_area = widgets.Output()

def on_ask_clicked(b):
    question = question_input.value.strip()
    if not question:
        return
    
    with output_area:
        print(f"\n{'='*60}")
        print(f"❓ Soru: {question}")
        print(f"{'='*60}")
        answer = rag_query(question, index, chunks_metadata, verbose=False)
        print(f"\n🤖 Cevap:\n{answer}")
        chat_history.append({"soru": question, "cevap": answer})
    
    question_input.value = ""

ask_button.on_click(on_ask_clicked)

print("💬 Ders Asistanı Hazır!")
display(widgets.VBox([
    widgets.HTML("<h3>🎓 Üniversite Ders Asistanı</h3>"),
    question_input,
    ask_button,
    output_area
]))

## 📊 13. Sistem İstatistikleri

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('RAG Sistem İstatistikleri', fontsize=14, fontweight='bold')

# 1. Chunk başına token dağılımı
token_counts = [c.token_count for c in chunks]
axes[0].hist(token_counts, bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Token/Chunk Dağılımı')
axes[0].set_xlabel('Token Sayısı')
axes[0].set_ylabel('Chunk Sayısı')

# 2. PDF başına chunk sayısı
file_counts = {}
for c in chunks:
    file_counts[c.source_file] = file_counts.get(c.source_file, 0) + 1
files = [f[:20] for f in file_counts.keys()]
counts = list(file_counts.values())
axes[1].barh(files, counts, color='coral', edgecolor='white')
axes[1].set_title('PDF Başına Chunk')
axes[1].set_xlabel('Chunk Sayısı')

# 3. NER varlık türleri
if all_labels:
    axes[2].bar(all_labels.keys(), all_labels.values(), color='mediumseagreen', edgecolor='white')
    axes[2].set_title('NER Varlık Türleri')
    axes[2].set_xlabel('Varlık Türü')
    axes[2].set_ylabel('Adet')

plt.tight_layout()
plt.savefig('rag_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik kaydedildi: rag_stats.png")

# Özet tablo
summary_data = {
    "Metrik": ["Toplam PDF", "Toplam Sayfa", "Toplam Chunk",
               "Ort. Token/Chunk", "Embedding Boyutu", "LLM Modeli"],
    "Değer":  [
        len(set(c.source_file for c in chunks)),
        len(parsed_pages),
        len(chunks),
        f"{np.mean(token_counts):.0f}",
        "1536D",
        CONFIG["llm_model"]
    ]
}
df_summary = pd.DataFrame(summary_data)
display(df_summary.to_string(index=False))

---
## ✅ Sistem Tamamlandı

| Adım | Araç | Açıklama |
|------|------|----------|
| PDF Parsing | PyMuPDF | Heading-aware sayfa çıkarma |
| Chunking | tiktoken | 300 token / 50 overlap |
| NER | spaCy | TOPIC, DATE, ORG, PERSON |
| Embedding | text-embedding-3-small | 1536 boyutlu vektör |
| Vector Store | FAISS IndexFlatIP | Cosine benzerlik |
| Generation | GPT-4o | Bağlam tabanlı cevap |

**GitHub:** Projenizi `git init` ile başlatın ve notebook'u push edin.